# Sentence Transformers

Устанавливаем фреймворк

In [2]:
! pip install -U sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 70.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 30.6 MB/s eta 0:00:00


Импортируем класс SentenceTransformer

In [3]:
from sentence_transformers import SentenceTransformer

Выбираем модель [из списка](https://www.sbert.net/docs/pretrained_models.html#multi-lingual-models) и создаем объект выбранной модели.

Замечание: я оставил якорную ссылку именно на мультиязычные модели. Специально для фреймворка Sentence Transformers существует приличное количество разных моделей, но почти все они обучены на англоязычных наборах данных. Все модели доступны на сайте Hugging Face, там же с помощью фильтра можно просматривать модели, совместимые с фреймворком Sentence Transformers.

Мы будем использовать [мультиязычную модель (15 языков)](https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1). Модель скачается с Hugging Face, можно заранее сохранить на диск. По умолчанию, если поддерживается CUDA, будет использоваться GPU. С помощью параметра device можно прямо указать, какой GPU использовать.

In [4]:
model = SentenceTransformer('distiluse-base-multilingual-cased-v1', device='cuda')

У каждой BERT-based модели есть максимальное количество токенов, которое мы можем подать на вход. Мы используем модель, которая основана на оригинальной модели DistillBERT, у которой ограничение 512 токенов.

In [5]:
model.max_seq_length # конкретно для этой модели такое значение

128

При этом видим, что сам фреймворк Sentence Transformers в целях оптимизации вычислений обрезает токены справа, вводя дополнительныей параметр - max_seq_length. Если вы хотите получать от моделей максимум, рекомендую ставить в качестве значения максимальное значение для базового трансформера, которое можно узнать в документации.

In [6]:
model.max_seq_length = 512

Если вы не хотите узнать, то можете ставить любые большие значения, пока не наткнетесь на ошибку, которая будет связана с позиционными эмбеддингами. Дело в том, что в предварительно обученных моделях конечно же заранее рассчитаны все позиционные эмбеддинги (в данной модели их 512). И если подадите больше токенов чем нужно, получится сложение тензоров различной размерности, что невозможно.

Получим, наконец, эмбеддинги!

In [7]:
sentences = ['Это фреймворк для получения эмбеддингов', 'Эмбеддинги - фундаментальная штука в NLP', 'Мне ничего не понятно, но очень интересно']

In [8]:
model.encode(sentences)

array([[ 0.01545597, -0.1051759 , -0.00030109, ..., -0.02971355,
        -0.00509099,  0.00522292],
       [-0.004131  ,  0.04195713, -0.02490227, ..., -0.01231962,
         0.01439469,  0.0006562 ],
       [ 0.02549078, -0.02064595, -0.01588603, ..., -0.03471693,
         0.03877511,  0.007195  ]], dtype=float32)

Мы получили эмбеддинги в виде массивом numpy, хотя использовали CUDA. Это сделано для максимизации совместимости. Но мы всегда можем получить тензоры PyTorch (причем, если используется CUDA, то они сразу же будут размещены на GPU).

In [9]:
model.encode(sentences, convert_to_numpy=False)

[tensor([ 1.5456e-02, -1.0518e-01, -3.0109e-04,  8.3220e-02,  6.6521e-02,
         -4.1430e-03, -4.0976e-02,  2.8297e-02, -3.4144e-02,  6.7970e-03,
         -9.0783e-02,  3.4358e-02, -1.2129e-03,  1.5628e-02,  1.8365e-02,
         -8.6732e-02,  3.0277e-02, -2.6940e-03,  1.2623e-02, -6.8880e-02,
          1.6368e-02,  4.6709e-02, -1.5277e-02, -2.9476e-02, -2.4311e-02,
         -4.4160e-02,  1.7765e-02,  4.3938e-03,  2.1355e-02,  1.5010e-02,
         -5.5136e-02, -1.5977e-02, -1.4673e-02, -2.4347e-02, -9.1043e-02,
          1.0195e-02, -3.6554e-02,  5.1672e-04, -8.8162e-04, -4.1755e-02,
         -5.1371e-02, -1.6224e-02,  5.6285e-02,  1.1389e-02,  5.8103e-02,
         -4.7658e-02, -7.7218e-03, -1.0218e-02, -5.9778e-02,  1.4761e-02,
         -3.8730e-02,  6.7005e-02, -5.1751e-03, -2.5474e-02,  3.5618e-02,
         -8.8383e-03, -3.5045e-02,  1.9913e-02, -1.8147e-02,  2.9630e-02,
         -2.7061e-02,  1.7718e-02,  1.8935e-02,  8.0364e-03, -6.2573e-03,
          3.5849e-02,  3.1537e-02,  4.

Обратите внимание, мы получили список тензоров. При этом можем получить один многомерный тензор, зачастую так удобно.

In [11]:
model.encode(sentences, convert_to_tensor=True)

tensor([[ 0.0155, -0.1052, -0.0003,  ..., -0.0297, -0.0051,  0.0052],
        [-0.0041,  0.0420, -0.0249,  ..., -0.0123,  0.0144,  0.0007],
        [ 0.0255, -0.0206, -0.0159,  ..., -0.0347,  0.0388,  0.0072]],
       device='cuda:0')

Посчитаем попарные значения схожести.

In [12]:
from sentence_transformers import util

In [13]:
embeddings = model.encode(sentences)
cosine_scores = util.cos_sim(embeddings, embeddings)
cosine_scores

tensor([[1.0000, 0.1962, 0.0061],
        [0.1962, 1.0000, 0.1023],
        [0.0061, 0.1023, 1.0000]])

In [14]:
embeddings[0].shape

(512,)

А вот так можно делать классификацию намерений!

In [15]:
q = ['Как зал в точке забронировать?']
i = ['Точка Кипения - это образовательно пространство, где можно проводить разные мероприятия', 'Для бронирования зала в Точке Кипения нужно написать мне', 'NLP - это объединение компьютерной лингвистики и искусственного интеллекта']

In [16]:
q_embeddings = model.encode(q)
i_embeddings = model.encode(i)
cosine_scores = util.cos_sim(q_embeddings, i_embeddings)
cosine_scores

tensor([[ 0.2079,  0.2407, -0.0771]])

Иногда следует приводить все в одному регистру!

In [17]:
q = ['как забукать точку?']
i = ['точка кипения - это образовательно пространство, где можно проводить разные мероприятия', 'Для бронирования зала в точке кипения нужно написать мне', 'nlp - это объединение компьютерной лингвистики и искусственного интеллекта']

In [18]:
q_embeddings = model.encode(q)
i_embeddings = model.encode(i)
cosine_scores = util.cos_sim(q_embeddings, i_embeddings)
cosine_scores

tensor([[ 0.2838,  0.3955, -0.0165]])

Сейчас вектора не нормированы.

In [19]:
sum(q_embeddings[0])

0.941865318978671

Рационально нормировать вектора и использовать скалярное произведение.

In [20]:
q_embeddings_norm = util.normalize_embeddings(q_embeddings)

AttributeError: ignored

Только это работает с тензорами, лол!

In [21]:
q_embeddings = model.encode(q, convert_to_numpy=False, convert_to_tensor=True)
i_embeddings = model.encode(i, convert_to_numpy=False, convert_to_tensor=True)
i_embeddings

tensor([[ 0.0573,  0.0126,  0.0438,  ..., -0.0301,  0.0276, -0.0284],
        [ 0.0419,  0.0797, -0.0164,  ...,  0.0049, -0.0186, -0.0110],
        [-0.0353, -0.0175, -0.0133,  ..., -0.0192,  0.0078, -0.0707]],
       device='cuda:0')

In [22]:
import torch

In [23]:
q_embeddings_norm = util.normalize_embeddings(q_embeddings)
i_embeddings_norm = util.normalize_embeddings(i_embeddings)
cosine_scores = util.dot_score(q_embeddings_norm, i_embeddings_norm)
cosine_scores

tensor([[ 0.2838,  0.3955, -0.0165]], device='cuda:0')

In [24]:
torch.topk(cosine_scores, 2)

torch.return_types.topk(
values=tensor([[0.3955, 0.2838]], device='cuda:0'),
indices=tensor([[1, 0]], device='cuda:0'))

Вообще рекомендую всегда работать с тензорами!

In [25]:
torch.topk(cosine_scores, 2).indices, torch.topk(cosine_scores, 2).values

(tensor([[1, 0]], device='cuda:0'),
 tensor([[0.3955, 0.2838]], device='cuda:0'))

Sentence Transformers - стандарт "зеленого банка" для получения векторных представлений текстовых документов.

Модели Sentence Transformers можно дообучить на своих данных, для этого [обратитесь к инструкции](https://www.sbert.net/docs/training/overview.html).